# Prepare a Qwen model dataset on Kaggle (CPU only)

Use this notebook to download the Hugging Face model snapshot directly into Kaggle notebook output, then create a private Kaggle Dataset from that output. It avoids transferring the multi-gigabyte snapshot through your local machine or using the dataset web UI's remote-URL importer. Select **Accelerator: None / CPU** and enable **Internet** in notebook options before starting. This download uses CPU notebook time and network bandwidth, but no GPU accelerator.

Kaggle's notebook and dataset UI may change. After the run completes, use the notebook's output/version page to create a private dataset from `qwen2.5-coder-3b-instruct`. Confirm the resulting dataset contains the model files at its root, or adjust `MODEL_DIR` in the inference notebook to the actual mounted subdirectory. Do not publish the dataset if you intend the model files to remain private.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "huggingface_hub"])
print("huggingface_hub is ready.")

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
MODEL_DIR = Path("/kaggle/working/qwen2.5-coder-3b-instruct")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id=MODEL_ID, local_dir=str(MODEL_DIR), ignore_patterns=["*.md", ".gitattributes"])
required_files = ("config.json", "tokenizer.json", "tokenizer_config.json")
missing_files = [name for name in required_files if not (MODEL_DIR / name).is_file()]
weight_files = list(MODEL_DIR.glob("*.safetensors"))
if missing_files or not weight_files:
    raise FileNotFoundError(f"Snapshot verification failed. Missing files: {missing_files}; safetensors found: {len(weight_files)}")
total_bytes = sum(path.stat().st_size for path in MODEL_DIR.rglob("*") if path.is_file())
print(f"Verified {MODEL_ID} at {MODEL_DIR}")
print(f"Safetensors shards: {len(weight_files)}; downloaded size: {total_bytes / 1024**3:.2f} GiB")

## Save as a private Kaggle Dataset

1. Run all cells and wait for snapshot verification to finish.
2. Use **Save Version** / commit so the notebook output is persisted.
3. From the completed notebook version's output, create a **private** dataset containing `qwen2.5-coder-3b-instruct`.
4. In the inference notebook, attach that dataset with **Add Input**, list `/kaggle/input` if needed, and set `MODEL_DIR` to the directory that directly contains `config.json` and the `.safetensors` shards.

The download cell is safe to rerun: `snapshot_download` reuses files already present in its local directory. Dataset creation and notebook output controls are managed by Kaggle; this cell does not call Kaggle APIs or require an API token.